The Required Files are avaliable in the Google Drive which can be accessed through Readme file in Github Repository

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, accuracy_score, precision_score,
                            recall_score, f1_score, roc_auc_score, roc_curve, auc)
from sklearn.base import BaseEstimator, ClassifierMixin
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import time
import datetime
import os
from google.colab import drive

In [2]:


#The Required Files are avaliable in the Google Drive which can be accessed through Readme file in Github Repository
# Data directory - using the processed data path
PROCESSED_DATA_PATH = 'nih_chestxray_augmented.csv' #nih_chestxray_augmented.csv can be found in the google drive link in the readme file
OUTPUT_DIR = 'xgboost_results'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

Mounted at /content/drive


In [3]:
class CustomXGBClassifier(BaseEstimator, ClassifierMixin):
    """Custom multi-label classifier using XGBoost with specified loss function."""
    def __init__(self, scale_pos_weights=None, **xgb_params):
        # Default XGBoost parameters optimized for A100 GPU
        self.default_params = {
            'objective': 'binary:logistic',  # Binary classification with log loss
            'eval_metric': 'logloss',        # Log loss for evaluation
            'tree_method': 'gpu_hist',       # GPU-optimized histogram algorithm
            'device': 'cuda',                # Use GPU
            'predictor': 'gpu_predictor',    # Use GPU for prediction
            'n_estimators': 300,             # Increased for better performance
            'max_depth': 8,                  # Can use deeper trees with GPU
            'learning_rate': 0.05,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 5,
            'gamma': 0.1,
            'reg_alpha': 0.1,
            'reg_lambda': 1.0,
            'verbosity': 1,
            'random_state': RANDOM_SEED
        }

        # Update with user params
        for k, v in xgb_params.items():
            self.default_params[k] = v

        self.xgb_params = self.default_params
        self.scale_pos_weights = scale_pos_weights
        self.estimators_ = []

    def fit(self, X, y):
        """Fits a separate XGBoost classifier for each label with custom weights."""
        self.estimators_ = []

        # Use tqdm for progress tracking
        for i, class_label in enumerate(tqdm(y.columns, desc="Training XGBoost models", unit="class")):
            print(f"\nTraining XGBoost for {class_label} ({i+1}/{len(y.columns)})...")

            # Create estimator with base parameters
            params = self.xgb_params.copy()

            # Set class-specific weight if available
            if self.scale_pos_weights is not None and class_label in self.scale_pos_weights:
                params['scale_pos_weight'] = self.scale_pos_weights[class_label]
                print(f"  - Using scale_pos_weight = {params['scale_pos_weight']:.2f}")

            # Create and train estimator
            estimator = XGBClassifier(**params)

            # Training progress tracking
            class_time_start = time.time()

            # Train with eval_set for monitoring
            eval_set = [(X, y.iloc[:, i])]  # Using training data for monitoring
            estimator.fit(X, y.iloc[:, i], eval_set=eval_set, verbose=50)

            # Report training completion
            class_time = time.time() - class_time_start
            print(f"  - Trained in {class_time:.2f} seconds ({class_time/60:.2f} minutes)")

            self.estimators_.append(estimator)

            # Display feature importance
            if hasattr(estimator, 'feature_importances_'):
                importances = estimator.feature_importances_
                indices = np.argsort(importances)[-5:]  # Top 5 features
                top_features = [X.columns[i] for i in indices[::-1]]
                print(f"  - Top features: {', '.join(top_features)}")

        return self

    def predict(self, X):
        """Make predictions for all labels."""
        n_samples = X.shape[0]
        n_labels = len(self.estimators_)
        y_pred = np.zeros((n_samples, n_labels), dtype=int)

        for i, estimator in enumerate(self.estimators_):
            y_pred[:, i] = estimator.predict(X)

        return y_pred

    def predict_proba(self, X):
        """Make probability predictions for all labels."""
        n_samples = X.shape[0]
        n_labels = len(self.estimators_)
        y_pred_proba = []

        for i, estimator in enumerate(self.estimators_):
            if hasattr(estimator, 'predict_proba'):
                y_pred_proba.append(estimator.predict_proba(X))
            else:
                # Fallback for estimators without predict_proba
                preds = estimator.predict(X)
                proba = np.zeros((n_samples, 2))
                proba[:, 1] = preds
                proba[:, 0] = 1 - preds
                y_pred_proba.append(proba)

        return y_pred_proba

In [4]:
def limit_samples_per_class(X, y, max_samples=2000):
    """
    Limit the number of samples per class to max_samples while maintaining ratio.

    Parameters:
    -----------
    X : DataFrame
        Feature data
    y : DataFrame
        Label data
    max_samples : int
        Maximum number of samples per class

    Returns:
    --------
    X_balanced : DataFrame
        Balanced feature data
    y_balanced : DataFrame
        Balanced label data
    """
    print(f"Limiting samples per class to maximum {max_samples}...")

    # Get the current counts for each class
    class_counts = y.sum().to_dict()
    print("Original class counts:")
    for label, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {label}: {count}")

    # Dictionary to keep track of which samples to keep for each class
    samples_to_keep = {}

    # For each class, randomly select samples to keep
    for label in y.columns:
        # Get positive samples for this class
        pos_indices = y[y[label] == 1].index.tolist()

        if len(pos_indices) <= max_samples:
            # Keep all samples if already under limit
            samples_to_keep[label] = pos_indices
            print(f"  - {label}: Keeping all {len(pos_indices)} samples (under max limit)")
        else:
            # Randomly select max_samples to keep
            np.random.seed(RANDOM_SEED + hash(label) % 1000)  # Unique seed for each label
            samples_to_keep[label] = np.random.choice(pos_indices, size=max_samples, replace=False).tolist()
            print(f"  - {label}: Reduced from {len(pos_indices)} to {max_samples} samples")

    # Combine all indices to keep
    all_indices_to_keep = set()
    for indices in samples_to_keep.values():
        all_indices_to_keep.update(indices)

    # Convert to sorted list
    all_indices_to_keep = sorted(list(all_indices_to_keep))

    # Create balanced dataset
    X_balanced = X.loc[all_indices_to_keep].reset_index(drop=True)
    y_balanced = y.loc[all_indices_to_keep].reset_index(drop=True)

    # Print resulting class counts
    print("\nClass counts after limiting to maximum of", max_samples, "per class:")
    for label in y.columns:
        count = y_balanced[label].sum()
        print(f"  - {label}: {count} ({count/len(y_balanced)*100:.2f}%)")

    print(f"\nTotal samples after limiting: {len(X_balanced)} (reduced from {len(X)})")

    return X_balanced, y_balanced

In [5]:
def stratified_multilabel_split(X, y, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    """
    Perform stratified split ensuring consistent class ratios across all splits.

    Parameters:
    -----------
    X : DataFrame
        Feature data
    y : DataFrame
        Label data
    train_ratio, val_ratio, test_ratio : float
        Ratio for train, validation, and test splits

    Returns:
    --------
    X_train, X_val, X_test, y_train, y_val, y_test
    """
    print("\nPerforming stratified split with ratios - Train: 8, Val: 1, Test: 1")
    print("This maintains the same class ratios across all splits.")

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-10, "Split ratios must sum to 1"

    # For each unique label combination, we'll split proportionally
    # First, get binary representation of each sample's labels
    y_binary = y.astype(int).astype(str).apply(''.join, axis=1)

    # Group samples by label combination
    combo_groups = {}
    for i, combo in enumerate(y_binary):
        if combo not in combo_groups:
            combo_groups[combo] = []
        combo_groups[combo].append(i)

    print(f"Found {len(combo_groups)} unique label combinations")

    # Prepare containers for split indices
    train_indices = []
    val_indices = []
    test_indices = []

    # For each combination, apply the split ratios
    for combo, indices in tqdm(combo_groups.items(), desc="Splitting data by label combination"):
        if len(indices) == 0:
            continue

        # Determine which labels are present in this combination
        present_labels = []
        for i, char in enumerate(combo):
            if char == '1':
                present_labels.append(y.columns[i])

        if len(present_labels) > 0:
            labels_str = ", ".join(present_labels)
        else:
            labels_str = "No labels (all zeros)"

        print(f"Splitting combination with labels: {labels_str} ({len(indices)} samples)")

        # Ensure reproducibility
        np.random.seed(RANDOM_SEED)
        np.random.shuffle(indices)

        # Calculate indices for each split
        n_total = len(indices)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)

        # Ensure at least 1 sample per split if possible
        if n_total >= 3:
            # Standard case - enough samples for all splits
            train_idx = indices[:n_train]
            val_idx = indices[n_train:n_train+n_val]
            test_idx = indices[n_train+n_val:]
            print(f"  - Split: Train {len(train_idx)}, Val {len(val_idx)}, Test {len(test_idx)}")
        elif n_total == 2:
            # Only 2 samples - prioritize train and validation
            train_idx = [indices[0]]
            val_idx = [indices[1]]
            test_idx = []
            print(f"  - Only 2 samples: Train 1, Val 1, Test 0")
        else:  # n_total == 1
            # Only 1 sample - place in training set
            train_idx = indices
            val_idx = []
            test_idx = []
            print(f"  - Only 1 sample: Train 1, Val 0, Test 0")

        # Add to our index lists
        train_indices.extend(train_idx)
        val_indices.extend(val_idx)
        test_indices.extend(test_idx)

    # Create the datasets
    X_train = X.iloc[train_indices].reset_index(drop=True)
    y_train = y.iloc[train_indices].reset_index(drop=True)

    X_val = X.iloc[val_indices].reset_index(drop=True)
    y_val = y.iloc[val_indices].reset_index(drop=True)

    X_test = X.iloc[test_indices].reset_index(drop=True)
    y_test = y.iloc[test_indices].reset_index(drop=True)

    # Print split information
    print(f"\nSplit results:")
    print(f"  - Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
    print(f"  - Validation set: {len(X_val)} samples ({len(X_val)/len(X)*100:.1f}%)")
    print(f"  - Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")

    # Validate that class ratios are preserved
    print("\nVerifying class distribution across splits:")
    for label in y.columns:
        total = y[label].sum()
        train = y_train[label].sum()
        val = y_val[label].sum()
        test = y_test[label].sum()

        train_pct = train/total*100 if total > 0 else 0
        val_pct = val/total*100 if total > 0 else 0
        test_pct = test/total*100 if total > 0 else 0

        print(f"  - {label}:")
        print(f"    * Total: {total} samples")
        print(f"    * Train: {train} samples ({train_pct:.1f}%)")
        print(f"    * Val: {val} samples ({val_pct:.1f}%)")
        print(f"    * Test: {test} samples ({test_pct:.1f}%)")

        # Check if the distribution is close to the expected ratios
        expected_train_pct = train_ratio * 100
        expected_val_pct = val_ratio * 100
        expected_test_pct = test_ratio * 100

        print(f"    * Expected ratio - Train: {expected_train_pct:.1f}%, Val: {expected_val_pct:.1f}%, Test: {expected_test_pct:.1f}%")

    return X_train, X_val, X_test, y_train, y_val, y_test

In [6]:
def find_optimal_thresholds(model, X_val, y_val):
    """Find optimal decision thresholds for each class using F1 score."""
    thresholds = {}

    # More granular threshold search
    threshold_range = np.arange(0.05, 0.95, 0.025)

    for i, label in enumerate(tqdm(y_val.columns,
                                  total=len(y_val.columns),
                                  desc="Optimizing thresholds",
                                  unit="label")):
        y_true = y_val.iloc[:, i]

        # Skip if all examples are the same class
        if y_true.nunique() <= 1:
            thresholds[label] = 0.5
            print(f"{label}: Using default threshold = 0.500 (insufficient class variety)")
            continue

        # Get predicted probabilities from the model
        y_proba = model.predict_proba(X_val)[i][:, 1]

        # Try different thresholds
        best_f1 = 0
        best_threshold = 0.5

        # Track all threshold results for plotting
        threshold_results = []

        for threshold in threshold_range:
            y_pred = (y_proba >= threshold).astype(int)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            threshold_results.append((threshold, f1))

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold

        thresholds[label] = best_threshold
        print(f"{label}: optimal threshold = {best_threshold:.3f}, F1 = {best_f1:.4f}")

        # Plot threshold curves for each label
        if threshold_results:
            plt.figure(figsize=(10, 5))
            thresholds_list, f1_scores = zip(*threshold_results)
            plt.plot(thresholds_list, f1_scores, 'b-', marker='o', markersize=4)
            plt.axvline(x=best_threshold, color='r', linestyle='--',
                       label=f'Best threshold: {best_threshold:.3f}')
            plt.title(f'F1 Score vs Threshold for {label}')
            plt.xlabel('Threshold')
            plt.ylabel('F1 Score')
            plt.grid(True)
            plt.legend()
            plt.savefig(f'{OUTPUT_DIR}/threshold_curve_{label}.png')
            plt.close()

    return thresholds

In [7]:
def predict_with_thresholds(model, X, thresholds):
    """Make predictions using custom thresholds."""
    y_pred_proba = model.predict_proba(X)
    n_samples = X.shape[0]
    n_labels = len(model.estimators_)
    y_pred = np.zeros((n_samples, n_labels), dtype=int)

    for i, label in enumerate(tqdm(thresholds.keys(),
                                  total=len(model.estimators_),
                                  desc="Generating predictions",
                                  unit="class")):
        # Get predicted probabilities
        y_proba = y_pred_proba[i][:, 1]

        # Apply custom threshold
        threshold = thresholds[label]
        y_pred[:, i] = (y_proba >= threshold).astype(int)

    return y_pred, y_pred_proba

In [8]:
def calculate_auc_roc(y_true, y_pred_proba, labels):
    """Calculate AUC-ROC for each label and plot ROC curves."""
    auc_scores = {}

    plt.figure(figsize=(12, 10))

    for i, label in enumerate(labels):
        y_true_label = y_true.iloc[:, i]

        # Skip labels with only one class
        if len(np.unique(y_true_label)) <= 1:
            print(f"Warning: Cannot calculate AUC for {label} - not enough class variety")
            continue

        # Get probabilities for positive class
        y_proba = y_pred_proba[i][:, 1]

        try:
            # Calculate ROC curve
            fpr, tpr, _ = roc_curve(y_true_label, y_proba)
            roc_auc = auc(fpr, tpr)
            auc_scores[label] = roc_auc

            # Plot ROC curve
            plt.plot(fpr, tpr, lw=1.5, label=f'{label} (AUC = {roc_auc:.2f})')
        except Exception as e:
            print(f"Error calculating AUC for {label}: {e}")
            continue

    # Plot diagonal line
    plt.plot([0, 1], [0, 1], 'k--', lw=1)

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for Multi-label Classification')
    plt.legend(loc="lower right", fontsize='small')
    plt.grid(True, alpha=0.3)
    plt.savefig(f'{OUTPUT_DIR}/roc_curves.png', dpi=300)
    plt.close()

    return auc_scores

In [9]:
def plot_metrics_summary(metrics_df, metric_name, output_path):
    """Plot summary metrics for all classes."""
    plt.figure(figsize=(12, 8))

    # Sort by value
    sorted_df = metrics_df.sort_values(by=metric_name, ascending=False)

    # Create bar plot
    ax = sns.barplot(x=metric_name, y='Label', data=sorted_df, palette='viridis')

    # Add value labels
    for i, v in enumerate(sorted_df[metric_name]):
        if isinstance(v, (int, float)):
            ax.text(max(0.01, v - 0.1), i, f"{v:.3f}", va='center')

    plt.title(f'{metric_name} for each disease class')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

In [10]:
def run_xgboost_implementation():
    """Run XGBoost implementation with processed data, class limiting, and 8:1:1 splitting."""
    start_time = time.time()

    print("="*100)
    print(f"NIH CHEST X-RAY CLASSIFICATION WITH XGBOOST (2000 SAMPLES PER CLASS)")
    print(f"Started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*100)

    # Step 1: Load the augmented data
    print("\nStep 1: Loading augmented data...")

    # Load the complete dataset
    df = pd.read_csv(PROCESSED_DATA_PATH)
    print(f"Loaded dataset with {len(df)} rows and {df.shape[1]} columns")

    # Separate features, labels, and metadata
    # Identify columns by type
    image_id_column = 'Image Index'
    label_columns = [col for col in df.columns if col in ['Fibrosis', 'Hernia', 'Nodule', 'Consolidation', 'No Finding',
                                                          'Atelectasis', 'Pneumonia', 'Cardiomegaly', 'Emphysema',
                                                          'Infiltration', 'Pleural_Thickening', 'Mass',
                                                          'Effusion', 'Pneumothorax', 'Edema']]
    feature_columns = [col for col in df.columns if col.startswith('feature_')]

    # Extract features and labels
    X = df[feature_columns]
    y = df[label_columns]

    # Display class distribution
    print("\nOriginal class distribution:")
    for label in label_columns:
        count = y[label].sum()
        print(f"  - {label}: {count} ({count/len(y)*100:.2f}%)")

    # Step 2: Limit samples per class to 2000
    print("\nStep 2: Limiting samples per class to 2000...")
    X_limited, y_limited = limit_samples_per_class(X, y, max_samples=2000)

    # Step 3: Split the data with 8:1:1 ratio while maintaining class distribution
    print("\nStep 3: Splitting data with 8:1:1 ratio...")
    X_train, X_val, X_test, y_train, y_val, y_test = stratified_multilabel_split(
        X_limited, y_limited, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1
    )

    # Step 4: Calculate class weights
    print("\nStep 4: Calculating scale_pos_weight values...")
    scale_pos_weights = {}

    for label in y_train.columns:
        num_neg = (y_train[label] == 0).sum()
        num_pos = (y_train[label] == 1).sum()
        weight = num_neg / max(1, num_pos)
        scale_pos_weights[label] = weight
        print(f"  - {label}: {num_pos} positives, {num_neg} negatives, scale_pos_weight = {weight:.2f}")

    # Step 5: Train XGBoost model
    print("\nStep 5: Training XGBoost models...")
    train_start = time.time()

    # Create XGBoost model with different objectives to try
    # Common objectives for classification: binary:logistic, binary:hinge, binary:logitraw
    xgb_params = {
        'objective': 'binary:logistic',  # Log loss objective
        'eval_metric': 'logloss',        # Log loss evaluation metric
        'tree_method': 'gpu_hist',       # Use GPU-accelerated algorithm
        'device': 'cuda'                 # Use CUDA for GPU acceleration
    }

    model = CustomXGBClassifier(scale_pos_weights=scale_pos_weights, **xgb_params)
    model.fit(X_train, y_train)

    train_time = time.time() - train_start
    print(f"Training completed in {train_time:.2f} seconds ({train_time/60:.2f} minutes)")

    # Step 6: Optimize thresholds
    print("\nStep 6: Optimizing decision thresholds...")
    thresholds = find_optimal_thresholds(model, X_val, y_val)

    # Step 7: Evaluate on test set
    print("\nStep 7: Evaluating model on test set...")

    # Generate standard predictions (threshold = 0.5)
    print("Generating standard threshold predictions...")
    y_pred_standard = model.predict(X_test)
    y_pred_standard_df = pd.DataFrame(y_pred_standard, columns=y_test.columns, index=y_test.index)

    # Generate optimized threshold predictions
    print("Generating optimized threshold predictions...")
    y_pred_optimized, y_pred_proba = predict_with_thresholds(model, X_test, thresholds)
    y_pred_optimized_df = pd.DataFrame(y_pred_optimized, columns=y_test.columns, index=y_test.index)

    # Calculate AUC-ROC scores
    print("\nCalculating AUC-ROC scores...")
    auc_scores = calculate_auc_roc(y_test, y_pred_proba, y_test.columns)

    # Calculate detailed metrics
    print("\nCalculating detailed metrics...")
    detailed_metrics = []

    # Calculate metrics for each class
    for i, label in enumerate(y_test.columns):
        # Get true and predicted values
        y_true = y_test.iloc[:, i]
        y_pred_std = y_pred_standard[:, i]
        y_pred_opt = y_pred_optimized[:, i]

        # Calculate standard threshold metrics
        accuracy_std = accuracy_score(y_true, y_pred_std)
        precision_std = precision_score(y_true, y_pred_std, zero_division=0)
        recall_std = recall_score(y_true, y_pred_std, zero_division=0)
        f1_std = f1_score(y_true, y_pred_std, zero_division=0)

        # Calculate optimized threshold metrics
        accuracy_opt = accuracy_score(y_true, y_pred_opt)
        precision_opt = precision_score(y_true, y_pred_opt, zero_division=0)
        recall_opt = recall_score(y_true, y_pred_opt, zero_division=0)
        f1_opt = f1_score(y_true, y_pred_opt, zero_division=0)

        # Get AUC score if available
        auc_score = auc_scores.get(label, None)

        # Store metrics
        detailed_metrics.append({
            'Label': label,
            'Support': y_true.sum(),
            'Accuracy (std)': accuracy_std,
            'Precision (std)': precision_std,
            'Recall (std)': recall_std,
            'F1 (std)': f1_std,
            'Accuracy (opt)': accuracy_opt,
            'Precision (opt)': precision_opt,
            'Recall (opt)': recall_opt,
            'F1 (opt)': f1_opt,
            'AUC': auc_score
        })

    # Create metrics DataFrame
    metrics_df = pd.DataFrame(detailed_metrics)

    # Print standard classification report
    print("\nStandard threshold (0.5) metrics:")
    print(classification_report(y_test, y_pred_standard_df))

    # Print optimized classification report
    print("\nOptimized threshold metrics:")
    print(classification_report(y_test, y_pred_optimized_df))

    # Print detailed metrics
    print("\nDetailed metrics for each disease:")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print(metrics_df)

    # Step 8: Plot metrics visualizations
    print("\nStep 8: Generating metrics visualizations...")

    # Plot F1 score comparison
    plt.figure(figsize=(14, 10))
    metrics_df_sorted = metrics_df.sort_values('F1 (opt)', ascending=False)

    bar_width = 0.35
    x = np.arange(len(metrics_df_sorted))

    fig, ax = plt.subplots(figsize=(14, 10))
    std_bars = ax.bar(x - bar_width/2, metrics_df_sorted['F1 (std)'], bar_width, label='Standard Threshold', color='blue', alpha=0.7)
    opt_bars = ax.bar(x + bar_width/2, metrics_df_sorted['F1 (opt)'], bar_width, label='Optimized Threshold', color='red', alpha=0.7)

    ax.set_xlabel('Disease Class')
    ax.set_ylabel('F1 Score')
    ax.set_title('F1 Score Comparison: Standard vs. Optimized Thresholds')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_df_sorted['Label'], rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/f1_score_comparison.png', dpi=300)
    plt.close()

    # Plot AUC values
    valid_auc = metrics_df[metrics_df['AUC'].notna()].sort_values('AUC', ascending=False)

    plt.figure(figsize=(12, 8))
    plt.barh(valid_auc['Label'], valid_auc['AUC'])
    plt.xlabel('AUC-ROC Score')
    plt.title('AUC-ROC Scores by Disease Class')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/auc_scores.png', dpi=300)
    plt.close()

    # Plot additional metrics
    plot_metrics_summary(metrics_df, 'Precision (opt)', f'{OUTPUT_DIR}/precision_summary.png')
    plot_metrics_summary(metrics_df, 'Recall (opt)', f'{OUTPUT_DIR}/recall_summary.png')
    plot_metrics_summary(metrics_df, 'Accuracy (opt)', f'{OUTPUT_DIR}/accuracy_summary.png')

    # Step 9: Save results and model
    print("\nStep 9: Saving results and model...")

    # Save metrics dataframe
    metrics_df.to_csv(f'{OUTPUT_DIR}/detailed_metrics.csv', index=False)

    # Save model
    with open(f'{OUTPUT_DIR}/xgboost_model.pkl', 'wb') as f:
        pickle.dump(model, f)

    # Save thresholds
    with open(f'{OUTPUT_DIR}/optimal_thresholds.pkl', 'wb') as f:
        pickle.dump(thresholds, f)

    # Calculate execution time
    total_time = time.time() - start_time

    # Print completion message
    print("\n"+"="*100)
    print(f"XGBOOST IMPLEMENTATION COMPLETED SUCCESSFULLY!")
    print(f"Total execution time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
    print(f"Finished at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*100)

    return {
        'model': model,
        'thresholds': thresholds,
        'metrics': metrics_df
    }

In [11]:
# Function to load model for prediction
def load_model_and_predict(image_features, model_path=None, thresholds_path=None):
    """
    Load the trained XGBoost model and make predictions on new data.

    Parameters:
    -----------
    image_features : DataFrame or numpy array
        Features extracted from chest X-ray images
    model_path : str, optional
        Path to the saved model file
    thresholds_path : str, optional
        Path to the saved thresholds file

    Returns:
    --------
    predictions : DataFrame
        Predicted disease labels
    """
    # Default paths
    if model_path is None:
        model_path = f'{OUTPUT_DIR}/xgboost_model.pkl'
    if thresholds_path is None:
        thresholds_path = f'{OUTPUT_DIR}/optimal_thresholds.pkl'

    # Load model and thresholds
    with open(model_path, 'rb') as f:
        model = pickle.load(f)

    with open(thresholds_path, 'rb') as f:
        thresholds = pickle.load(f)

    # Convert input to DataFrame if necessary
    if not isinstance(image_features, pd.DataFrame):
        # Load feature columns to get column names
        with open(f'{DATA_DIR}/feature_columns.pkl', 'rb') as f:
            feature_columns = pickle.load(f)

        # Convert to DataFrame
        image_features = pd.DataFrame(image_features, columns=feature_columns)

    # Make predictions with optimal thresholds
    predictions = predict_with_thresholds(model, image_features, thresholds)

    # Convert to DataFrame
    with open(f'{DATA_DIR}/label_columns.pkl', 'rb') as f:
        label_columns = pickle.load(f)

    predictions_df = pd.DataFrame(predictions, columns=label_columns)

    return predictions_df

In [12]:
if __name__ == "__main__":
    results = run_xgboost_implementation()
    print("XGBoost implementation completed!")

NIH CHEST X-RAY CLASSIFICATION WITH XGBOOST (2000 SAMPLES PER CLASS)
Started at: 2025-04-23 03:25:26

Step 1: Loading augmented data...
Loaded dataset with 97257 rows and 1042 columns

Original class distribution:
  - Fibrosis: 6520 (6.70%)
  - Hernia: 2287 (2.35%)
  - Nodule: 17837 (18.34%)
  - Consolidation: 16252 (16.71%)
  - No Finding: 9133 (9.39%)
  - Atelectasis: 29928 (30.77%)
  - Pneumonia: 6754 (6.94%)
  - Cardiomegaly: 9135 (9.39%)
  - Emphysema: 8875 (9.13%)
  - Infiltration: 43274 (44.49%)
  - Pleural_Thickening: 12917 (13.28%)
  - Mass: 18016 (18.52%)
  - Effusion: 34856 (35.84%)
  - Pneumothorax: 15241 (15.67%)
  - Edema: 8861 (9.11%)

Step 2: Limiting samples per class to 2000...
Limiting samples per class to maximum 2000...
Original class counts:
  - Infiltration: 43274
  - Effusion: 34856
  - Atelectasis: 29928
  - Mass: 18016
  - Nodule: 17837
  - Consolidation: 16252
  - Pneumothorax: 15241
  - Pleural_Thickening: 12917
  - Cardiomegaly: 9135
  - No Finding: 9133
  

Splitting data by label combination:   0%|          | 0/2287 [00:00<?, ?it/s]

Splitting combination with labels: Cardiomegaly, Effusion (146 samples)
  - Split: Train 116, Val 14, Test 16
Splitting combination with labels: Hernia (417 samples)
  - Split: Train 333, Val 41, Test 43
Splitting combination with labels: Hernia, Infiltration (74 samples)
  - Split: Train 59, Val 7, Test 8
Splitting combination with labels: No Finding (2000 samples)
  - Split: Train 1600, Val 200, Test 200
Splitting combination with labels: Atelectasis (289 samples)
  - Split: Train 231, Val 28, Test 30
Splitting combination with labels: Mass, Effusion (79 samples)
  - Split: Train 63, Val 7, Test 9
Splitting combination with labels: Emphysema, Infiltration, Pleural_Thickening, Pneumothorax (25 samples)
  - Split: Train 20, Val 2, Test 3
Splitting combination with labels: Infiltration, Mass, Pneumothorax (9 samples)
  - Split: Train 7, Val 0, Test 2
Splitting combination with labels: Nodule, Cardiomegaly, Infiltration, Mass (15 samples)
  - Split: Train 12, Val 1, Test 2
Splitting comb

Training XGBoost models:   0%|          | 0/15 [00:00<?, ?class/s]


Training XGBoost for Fibrosis (1/15)...
  - Using scale_pos_weight = 6.97


...

Generating predictions:   0%|          | 0/15 [00:00<?, ?class/s]


Calculating AUC-ROC scores...

Calculating detailed metrics...

Standard threshold (0.5) metrics:
              precision    recall  f1-score   support

           0       0.71      0.45      0.55       489
           1       0.83      0.82      0.83       265
           2       0.71      0.53      0.61       970
           3       0.66      0.52      0.58       911
           4       0.54      0.52      0.53       200
           5       0.70      0.66      0.68      1350
           6       0.76      0.38      0.50       507
           7       0.75      0.63      0.69       574
           8       0.76      0.57      0.65       600
           9       0.70      0.68      0.69      1716
          10       0.69      0.50      0.58       858
          11       0.74      0.66      0.70      1044
          12       0.74      0.75      0.75      1585
          13       0.73      0.59      0.65       838
          14       0.69      0.64      0.66       581

   micro avg       0.72      0.61  

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
<ipython-input-9-e16feef9947d>:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=metric_name, y='Label', data=sorted_df, palette='viridis')
<ipython-input-9-e16feef9947d>:9: FutureWarning: 

Passing `palette` 


Step 9: Saving results and model...

XGBOOST IMPLEMENTATION COMPLETED SUCCESSFULLY!
Total execution time: 370.93 seconds (6.18 minutes)
Finished at: 2025-04-23 03:31:37
XGBoost implementation completed!


<Figure size 1400x1000 with 0 Axes>